# HW1 · Part 2. Zhang 캘리브레이션 — scratch 구현 (30점 · 예상 6–8시간)

**배포본:** v5 (2026-10-20)

**목표** — 체커보드 세트에서 K · 왜곡 k₁ k₂ · 자세를 직접 구하기: 닫힌 해(Vb = 0) → 왜곡 선형 초기화 → LM 정제. 결과를 OpenCV 와 비교하고, 세트 A · B · C 의 차이로 "RMS 가 낮다 ≠ 좋다" 를 본인 데이터로 확인

**참고** — Zhang 2000 §2–4 · Appendix B · C (homography 제약 · 닫힌 해 · ML 정제 · 왜곡 · 퇴화), MVG 2판 §7.2 (기하 오차 · Algorithm 7.1 Gold Standard) · Result 5.2 (공분산), OpenCV `calibrateCamera` 문서

**TODO** — `v_ij` · `solve_B` · `K_from_B` · `extrinsics_from_H` · `init_distortion_linear` · `distort_normalized` · `project_points` · `residuals` · `covariance_from_jacobian` (9개)

**OpenCV 허용** — `cv2.findHomography`(2.1), `cv2.Rodrigues`(제공 `geometry.rodrigues` 대신 써도 됨), 2.7 비교용 `cv2.calibrateCameraExtended`. **그 외 OpenCV 캘리브레이션 함수로 TODO 를 대체하면 0점**(fallback 은 맨 아래 규정대로 감점 후 허용).

> **허용 · 금지** — NumPy · SciPy · Matplotlib 자유. OpenCV 는 각 Part 안내에 "허용" 으로 적힌 함수만(코너 검출 · `findHomography` · 검증용 비교). 
> `TODO` 로 표시된 함수를 OpenCV 호출로 대체하면 해당 항목 0점. 

In [ ]:
# ---- 공통 설정 (HW1 v5 · 모든 노트북에서 같은 설정 파일 사용) ----
import os, sys, json, glob, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'hw1_config.json').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'hw1_config.json').is_file():
    raise FileNotFoundError('HW1 폴더 또는 notebooks 폴더에서 실행하세요: hw1_config.json 을 찾을 수 없습니다.')
sys.path.insert(0, str(ROOT))
from hw1lib import geometry as G, synth as S, exif as E, board as Bd, io_utils as IO, checks as CK, click_tool as CT
# 데모 전환은 HW1/hw1_config.json 의 mode 만 "demo" 로 변경하고 각 노트북의 이 셀을 다시 실행.
cfg = IO.load_config(ROOT)
DATA, RESULTS = cfg['DATA'], cfg['RESULTS']
IO.RESULTS = RESULTS
def load_corners_checked(name):
    """현재 설정과 저장 코너 파일의 검출 배율이 같은지 확인."""
    det = Bd.load_corners(IO.rpath(name))
    if not np.isclose(det['scale'], cfg['SCALE'], rtol=0, atol=1e-12):
        raise ValueError(f"{name}: 저장 배율 {det['scale']}과 현재 설정 scale={cfg['SCALE']}이 다릅니다. Part 0부터 다시 실행하세요.")
    return det
os.makedirs(RESULTS, exist_ok=True)
# 서로 다른 데이터의 결과를 같은 폴더에 섞지 않음. 상대 경로로 기록하여 제출 폴더 이동은 허용.
context = dict(mode=cfg['mode'], data=os.path.relpath(DATA, ROOT), version=5)
context_path = IO.rpath('dataset_context.json')
if os.path.exists(context_path):
    if IO.load_json(context_path) != context:
        raise ValueError('결과 폴더의 데이터 출처가 현재 설정과 다릅니다. 새 results 경로를 지정하고 Part 0부터 실행하세요.')
elif any(not p.name.startswith('.') for p in Path(RESULTS).iterdir()):
    raise ValueError('출처 기록이 없는 기존 결과가 있습니다. 새 results 경로를 지정하고 Part 0부터 실행하세요.')
IO.save_json(context_path, context)
np.set_printoptions(precision=4, suppress=True)
import matplotlib
_fonts = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Freesentation']:
    if _f in _fonts:
        matplotlib.rcParams['font.family'] = _f; break
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore', message='Glyph')
%matplotlib inline
print('mode =', cfg['mode'], '\nDATA =', DATA, '\nRESULTS =', RESULTS)


## 2.1 코너 · 이미지별 H (제공)
Part 0 에서 저장한 코너를 읽고, 장마다 보드 (X, Y) mm → 픽셀 의 H 를 `cv2.findHomography` 로 구합니다 (H 추정 자체는 5주차).

In [ ]:
import cv2
dets = {s: load_corners_checked(f'corners_{s}.npz') for s in ['setA', 'setB', 'setC']}
def homographies(det):
    return [cv2.findHomography(X[:, :2].astype(np.float64), m.astype(np.float64))[0] for X, m in zip(det['objpoints'], det['imgpoints'])]
Hs = {s: homographies(d) for s, d in dets.items()}
for s in dets:
    print(s, len(Hs[s]), '장, image_size', dets[s]['image_size'])
detC = dets['setC']; W, H = detC['image_size']

## 2.2 닫힌 해 — v_ij → Vb = 0 → B → K (6점)
평면 한 장의 H 가 ω = B = K⁻ᵀK⁻¹ 에 주는 제약 두 개(Zhang §2.3)를 모아 b 를 풀고(§3.1), Appendix B 의 닫힌 식으로 K 를 꺼냅니다.

In [ ]:
# ==============================================================================
# TODO  v_ij
# 참조: Zhang 2000 §3.1 식 (7)–(8)
# h_i = H 의 i 번째 "열" (h_i1, h_i2, h_i3).  B 는 대칭이므로 b = [B11, B12, B22, B13, B23, B33] 6-벡터
# h_iᵀ B h_j = v_ijᵀ b,   v_ij = [h_i1 h_j1,  h_i1 h_j2 + h_i2 h_j1,  h_i2 h_j2,  h_i3 h_j1 + h_i1 h_j3,  h_i3 h_j2 + h_i2 h_j3,  h_i3 h_j3]
# 인덱스 i, j 는 0 부터 (h_1 → H[:, 0]).  반환: (6,) 배열
# ==============================================================================
def v_ij(H, i, j):
    """Zhang 식 (7)–(8): v_ij = [h_i1 h_j1, h_i1 h_j2 + h_i2 h_j1, h_i2 h_j2, h_i3 h_j1 + h_i1 h_j3, h_i3 h_j2 + h_i2 h_j3, h_i3 h_j3]
    h_i 는 H 의 i 번째 '열' (Zhang §3.1 닫힌 해)."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: v_ij')


# ==============================================================================
# TODO  solve_B
# 참조: Zhang 2000 §3.1 식 (9), MVG A5.3 (‖Ap‖ 최소화 — 답은 최소 특이값의 우특이벡터)
# 한 장의 H 가 주는 제약 두 개 (Zhang §2.3 식 (3)–(4)): h1ᵀ B h2 = 0,  h1ᵀ B h1 = h2ᵀ B h2  →  v12ᵀ b = 0,  (v11 − v22)ᵀ b = 0
# n 장을 쌓으면 V (2n × 6).  Vb = 0 의 최소제곱 해 = Vᵀ V 의 최소 고유벡터 = SVD 의 최소 특이값에 대응하는 우특이벡터 (np.linalg.svd → Vt[-1])
# b → 대칭 행렬 B 로 재배열.  반환: (B (3×3), 특이값 s (6,))  — s[-1]/s[-2] 가 작을수록 제약이 잘 결정됨 (세트 A 와 비교해 볼 것)
# ==============================================================================
def solve_B(Hs):
    """여러 장의 H 에서 B = K⁻ᵀK⁻¹ (대칭 3×3)  — Vb = 0, b = 최소 특이값의 우특이벡터 (Zhang §3.1 식 (9), MVG A5.3).
    각 H 는 2행: v12ᵀ b = 0,  (v11 − v22)ᵀ b = 0."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: solve_B')


# ==============================================================================
# TODO  K_from_B
# 참조: Zhang 2000 Appendix B (닫힌 식 5개)
# v0 = (B12 B13 − B11 B23) / (B11 B22 − B12²)
# λ  = B33 − [B13² + v0 (B12 B13 − B11 B23)] / B11
# α  = sqrt(λ / B11),   β = sqrt(λ B11 / (B11 B22 − B12²)),   γ = −B12 α² β / λ,   u0 = γ v0 / β − B13 α² / λ
#    ※ Zhang 원문 Appendix B 의 u0 식은 "γ v0 / α" 로 인쇄되어 있으나 오타 — B13 = λ(γ v0 − u0 β)/(α² β) 를 u0 에 대해 풀면 β 로 나눠야 함 (γ = 0 이면 차이 없음)
# B 는 스케일이 임의(−B 도 해) → B11 < 0 이면 B ← −B.  sqrt 안이 음수면 "B 가 양정치가 아님" = 닫힌 해 실패 → ValueError 로 알릴 것 (세트 A 에서 일어남 — Zhang §4 Proposition 1 의 퇴화)
# 반환: K = [[α, γ, u0], [0, β, v0], [0, 0, 1]]  (geometry.make_K(α, β, u0, v0, γ))
# ==============================================================================
def K_from_B(B):
    """Zhang Appendix B: B 에서 (α, β, γ, u0, v0) 닫힌 식.  B 가 음정치이면 −B 로.
    ※ u0 = γ v0/β − B13 α²/λ.  Zhang 논문 원문(Appendix B)의 γ v0/α 는 오타 — B13 = λ(γ v0 − u0 β)/(α² β) 에서 풀면 β 로 나눠야 함 (γ = 0 이면 차이 없음)."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: K_from_B')


In [ ]:
B, sv = solve_B(Hs['setC'])
K0 = K_from_B(B)
print('특이값 (V):', sv, ' 최소/차소 =', sv[-1] / sv[-2])
print('닫힌 해 K0 (세트 C):\n', K0)
CK.check_closed_form(solve_B, K_from_B, noise_px=0.0); CK.check_closed_form(solve_B, K_from_B, noise_px=0.3);

## 2.3 외부 파라미터 — 장마다 (R, t) (3점)
Part 1.2 의 분해를 그대로 씁니다 (λ = 1/‖K⁻¹h₁‖, r₃ = r₁ × r₂, SVD 직교화 — Appendix C).

In [ ]:
# ==============================================================================
# TODO  extrinsics_from_H
# Part 1.2 의 decompose_board_homography 를 여기에 복사 (같은 함수)
# ==============================================================================
def extrinsics_from_H(K, H):
    """각 장의 (R, t) — Part 1.2 의 decompose_board_homography 와 같음 (Zhang §3.1 끝 · Appendix C 직교화).
    λ = 1/‖K⁻¹h1‖, r1 = λK⁻¹h1, r2 = λK⁻¹h2, r3 = r1 × r2, t = λK⁻¹h3.  t_z < 0 이면 부호 반전.  R 은 SVD 로 직교화."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: extrinsics_from_H')


In [ ]:
K0s = K0.copy(); K0s[0, 1] = 0.0            # skew 는 0 으로 고정 (제약 카메라, MVG §7.3)
Rs0, ts0 = zip(*[extrinsics_from_H(K0s, Hh) for Hh in Hs['setC']])
Rs0, ts0 = list(Rs0), list(ts0)
print('t_z (보드 원점의 깊이, mm) 처음 5장:', [round(float(t[2]), 1) for t in ts0[:5]], ' 모두 양수?', all(t[2] > 0 for t in ts0))
CK.check_extrinsics(extrinsics_from_H);

## 2.4 왜곡 k₁ k₂ 선형 초기화 — Zhang 식 (13) (3점)

In [ ]:
# ==============================================================================
# TODO  init_distortion_linear
# 참조: Zhang 2000 §3.3 식 (11)–(13)
# 닫힌 해의 K · (R, t) 로 각 점의 "이상" 투영 (u, v) (왜곡 없음) 과 정규화 좌표 (x, y), r² = x² + y² 를 계산
# 관측 (ŭ, v̆) 과의 차이가 방사 왜곡: ŭ − u = (u − u0)(k1 r² + k2 r⁴),  v̆ − v = (v − v0)(k1 r² + k2 r⁴)
# → 모든 점 · 모든 장을 쌓아 D k = d (2N × 2) 를 최소제곱 (np.linalg.lstsq).  반환: dist = [k1, k2, 0, 0, 0]
# ※ 이 식은 K 가 이미 맞다고 가정 — 그래서 "초기화" 이고 최종값은 LM 이 정함
# ==============================================================================
def init_distortion_linear(K, Rs, ts, objpoints, imgpoints):
    """k1, k2 선형 초기화 — Zhang §3.3 식 (13):
       [(u−u0) r², (u−u0) r⁴ ; (v−v0) r², (v−v0) r⁴] [k1 k2]ᵀ = [ŭ−u ; v̆−v]
    (u, v) 이상 투영, (ŭ, v̆) 관측, r² 는 정규화 좌표의 제곱 반지름."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: init_distortion_linear')


In [ ]:
dist0 = init_distortion_linear(K0s, Rs0, ts0, detC['objpoints'], detC['imgpoints'])
print('초기 k1, k2 =', dist0[:2])
CK.check_init_distortion(init_distortion_linear);

## 2.5 LM 정제 — 투영 함수 · 잔차 · 공분산 (9점)
파라미터 p = [fx, fy, cx, cy, k₁, k₂, rvec₁, tvec₁, …] 전부를 `scipy.optimize.least_squares(method='lm')` 로 정제합니다 (MVG Algorithm 7.1 Gold Standard 의 2단계, Zhang §3.2–3.3).
`pack_params` · `unpack_params` · `refine_lm` 은 제공. 여러분은 **투영 함수**(좌표 변환 사슬), **잔차**(기하 오차), **공분산**(Result 5.2)을 작성합니다.

In [ ]:
# ==============================================================================
# TODO  distort_normalized
# 참조: Brown–Conrady 왜곡 모델 (OpenCV calib3d 문서, 계수 순서 k1, k2, p1, p2, k3)
# r² = x² + y²,  방사 = 1 + k1 r² + k2 r⁴ + k3 r⁶
# x_d = x · 방사 + 2 p1 x y + p2 (r² + 2x²),   y_d = y · 방사 + p1 (r² + 2y²) + 2 p2 x y
# 정규화 좌표(K 적용 전)에서 정의됨 — 픽셀 좌표에 적용하면 안 됨 (Part 6 함정 문항).  배열 입력 지원
# ==============================================================================
def distort_normalized(xn, yn, dist):
    """Brown–Conrady 순방향 (정규화 좌표) — OpenCV 순서 (k1, k2, p1, p2, k3)."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: distort_normalized')


# ==============================================================================
# TODO  project_points
# 참조: MVG §6.1 투영 사슬, Zhang 2000 §3.2–3.3 식 (10)–(12)
# 사슬: X_c = R X_w + t → (x, y) = (X_c/Z_c, Y_c/Z_c) → distort_normalized → (u, v) = (fx x_d + s y_d + cx, fy y_d + cy)
# skew s = K[0,1] 도 포함 (보통 0).  반환 (N, 2).  합성 검증: checks.check_project_points
# ==============================================================================
def project_points(K, dist, R, t, Xw):
    """Xw (N, 3) → (N, 2) 픽셀, 왜곡 포함. 사슬: [R|t] → ÷Z → D(·) → K  (Zhang §3.2–3.3 식 (10)–(12))."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: project_points')


In [ ]:
# 제공 — 파라미터 벡터 ↔ (K, dist, Rs, ts)
def pack_params(K, dist, Rs, ts, fix_skew=True, n_dist=2):
    """파라미터 벡터 p = [fx, fy, cx, cy, (skew), k1, k2, (p1, p2), rvec_1, tvec_1, …]"""
    p = [K[0, 0], K[1, 1], K[0, 2], K[1, 2]]
    if not fix_skew:
        p.append(K[0, 1])
    p += list(np.ravel(dist)[:n_dist])
    for R, t in zip(Rs, ts):
        p += list(G.inv_rodrigues(R)); p += list(np.ravel(t))
    return np.array(p, float)


def unpack_params(p, n_views, fix_skew=True, n_dist=2):
    i = 0
    fx, fy, cx, cy = p[0:4]; i = 4
    skew = 0.0
    if not fix_skew:
        skew = p[i]; i += 1
    dist = np.zeros(5); dist[:n_dist] = p[i:i + n_dist]; i += n_dist
    K = G.make_K(fx, fy, cx, cy, skew)
    Rs, ts = [], []
    for _ in range(n_views):
        Rs.append(G.rodrigues(p[i:i + 3])); ts.append(p[i + 3:i + 6].copy()); i += 6
    return K, dist, Rs, ts


In [ ]:
# ==============================================================================
# TODO  residuals
# 참조: MVG §7.2 기하 오차 (측정점과 재투영점 사이의 픽셀 거리), Zhang 2000 식 (10)/(14)
# unpack_params(p, n_views) 로 K, dist, Rs, ts 를 풀고, 장마다 project_points(K, dist, R_i, t_i, X_i) − m_i 를 (N_i·2,) 로 펴서 이어 붙임
# 반환 (2 N_total,) — scipy.optimize.least_squares 가 이 벡터의 제곱합을 최소화 (LM — MVG Algorithm 7.1 Gold Standard 의 2단계)
# ==============================================================================
def residuals(p, objpoints, imgpoints, fix_skew=True, n_dist=2):
    """재투영 잔차 벡터 (2 N_total,)  — Zhang 식 (10)/(14) 의 합을 이루는 항  (MVG §7.2 기하 오차)."""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: residuals')


# ==============================================================================
# TODO  covariance_from_jacobian
# 참조: MVG Result 5.2 (잔차에서 σ, σ에서 Σ ≈ σ²(JᵀJ)⁻¹)
# σ² = rᵀr / (m − n)  (m = 잔차 개수, n = 파라미터 개수),  Σ = σ² (JᵀJ)⁻¹  (역행렬이 불안정하면 np.linalg.pinv)
# 반환: (Σ, σ).  np.sqrt(np.diag(Σ))[:4] 가 (fx, fy, cx, cy) 의 표준편차 — OpenCV stdDeviationsIntrinsics 와 같은 정의
# ==============================================================================
def covariance_from_jacobian(J, r):
    """σ² = rᵀr/(m − n),  Σ = σ² (JᵀJ)⁻¹  (MVG Result 5.2).  반환: Σ, σ"""
    # ---- 여기에 작성 ----
    raise NotImplementedError('TODO: covariance_from_jacobian')


In [ ]:
# 제공 — least_squares 호출 래퍼 · RMS (OpenCV 정의)
def refine_lm(K0, dist0, Rs0, ts0, objpoints, imgpoints, fix_skew=True, n_dist=2):
    """scipy least_squares (LM) 로 전체 파라미터 정제. 반환: K, dist, Rs, ts, result(J 포함)."""
    from scipy.optimize import least_squares
    p0 = pack_params(K0, dist0, Rs0, ts0, fix_skew, n_dist)
    res = least_squares(residuals, p0, args=(objpoints, imgpoints, fix_skew, n_dist), method='lm', xtol=1e-12, ftol=1e-12, max_nfev=2000)
    if not res.success:
        warnings.warn(f'LM이 수렴 조건을 만족하지 못했습니다: {res.message}. 결과와 σ를 확정값으로 해석하지 마세요.', RuntimeWarning)
    K, dist, Rs, ts = unpack_params(res.x, len(objpoints), fix_skew, n_dist)
    return K, dist, Rs, ts, res


def rms_reprojection(K, dist, Rs, ts, objpoints, imgpoints):
    """OpenCV 정의의 RMS: sqrt(mean(‖m − m̂‖²)) over all points."""
    e = []
    for R, t, X, m in zip(Rs, ts, objpoints, imgpoints):
        e.append(np.sum((project_points(K, dist, R, t, X) - m) ** 2, axis=1))
    return float(np.sqrt(np.mean(np.concatenate(e))))


In [ ]:
import time
t0 = time.time()
K_z, dist_z, Rs, ts, res = refine_lm(K0s, dist0, Rs0, ts0, detC['objpoints'], detC['imgpoints'])
Sigma, sigma = covariance_from_jacobian(res.jac, res.fun)
std = np.sqrt(np.diag(Sigma))
rms = rms_reprojection(K_z, dist_z, Rs, ts, detC['objpoints'], detC['imgpoints'])
per_view = [float(np.sqrt(np.mean(np.sum((project_points(K_z, dist_z, R, t, X) - m) ** 2, axis=1)))) for R, t, X, m in zip(Rs, ts, detC['objpoints'], detC['imgpoints'])]
print('LM 수렴:', res.success, '| 종료 이유:', res.message)
print(f'LM {res.nfev} 회 평가, {time.time()-t0:.1f} s,  RMS {rms:.4f} px,  σ(잔차) {sigma:.4f} px')
print('K (세트 C, scratch):\n', K_z, '\ndist =', dist_z[:2])
print('σ(fx, fy, cx, cy, k1, k2) =', std[:6])
print('장별 RMS:', np.round(per_view, 3))
def save_zhang(name, K, dist, std, rms, per_view, K0, det, extra=None):
    d = dict(K=K, dist=dist, rms=rms, per_view_rms=per_view, std=dict(fx=std[0], fy=std[1], cx=std[2], cy=std[3], k1=std[4], k2=std[5]),
             K0=K0, n_images=len(det['names']), image_size=list(det['image_size']), scale=det['scale'], names=det['names'], source='scratch')
    if extra: d.update(extra)
    IO.save_json(IO.rpath(name), d)
save_zhang('zhang_setC.json', K_z, dist_z, std, rms, per_view, K0, detC, extra=dict(converged=bool(res.success), termination=res.message))
IO.save_K('K.json', K_z, (W, H), 'zhang_scratch_setC', converged=bool(res.success), termination=res.message, std=dict(fx=std[0], fy=std[1], cx=std[2], cy=std[3]), rms=rms)
IO.save_json(IO.rpath('dist.json'), dict(dist=dist_z, model='opencv_k1k2', std=dict(k1=std[4], k2=std[5]), source='zhang_scratch_setC'))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
resid = [m - project_points(K_z, dist_z, R, t, X) for R, t, X, m in zip(Rs, ts, detC['objpoints'], detC['imgpoints'])]
Bd.residual_map(detC['imgpoints'], resid, (W, H), ax=ax[0], title='세트 C 잔차 (관측 - 재투영)')
ax[1].bar(range(len(per_view)), per_view); ax[1].set_xlabel('image'); ax[1].set_ylabel('RMS (px)'); ax[1].set_title('장별 RMS')
fig.tight_layout(); fig.savefig(IO.rpath('part2_residuals_setC.png'), dpi=110)
CK.check_project_points(project_points); CK.check_residuals(residuals, pack_params); CK.check_covariance(covariance_from_jacobian);

## 2.6 합성 검증 — 참값을 아는 데이터로 전체 파이프라인 (3점)
`hw1lib.synth` 가 만든 세트(참 K · k · 자세)로 전체 파이프라인을 돌려 복원 오차를 표로 만듭니다 (Zhang 2000 §5.1 합성 실험의 축소판).
실데이터보다 **먼저** 통과해야 합니다 — 여기서 틀리면 뒤의 모든 숫자가 의미가 없습니다.

In [ ]:
def my_zhang(objpoints, imgpoints, image_size, K_init=None):
    """전체 파이프라인 — 세트 A 처럼 닫힌 해가 실패하면 K_init 으로 시작."""
    import cv2
    Hs_ = [cv2.findHomography(X[:, :2].astype(np.float64), m.astype(np.float64))[0] for X, m in zip(objpoints, imgpoints)]
    B_, sv_ = solve_B(Hs_)
    if K_init is None:
        K0_ = K_from_B(B_)
    else:
        K0_ = np.asarray(K_init, float).copy()
    K0_[0, 1] = 0.0
    Rs_, ts_ = map(list, zip(*[extrinsics_from_H(K0_, Hh) for Hh in Hs_]))
    d0_ = init_distortion_linear(K0_, Rs_, ts_, objpoints, imgpoints)
    K_, d_, Rs_, ts_, res_ = refine_lm(K0_, d0_, Rs_, ts_, objpoints, imgpoints)
    Sigma_, sigma_ = covariance_from_jacobian(res_.jac, res_.fun)
    pv = [float(np.sqrt(np.mean(np.sum((project_points(K_, d_, R, t, X) - m) ** 2, axis=1)))) for R, t, X, m in zip(Rs_, ts_, objpoints, imgpoints)]
    return dict(converged=bool(res_.success), termination=res_.message, K=K_, dist=d_, Rs=Rs_, ts=ts_, rms=rms_reprojection(K_, d_, Rs_, ts_, objpoints, imgpoints), std=np.sqrt(np.diag(Sigma_)), per_view_rms=pv, K0=K0_, sv=sv_)

print('참값: f = %.1f / %.1f, c = (%.1f, %.1f), k1 k2 = %s' % (S.K_TRUE[0, 0], S.K_TRUE[1, 1], S.K_TRUE[0, 2], S.K_TRUE[1, 2], S.DIST_TRUE[:2]))
print('%-8s %-6s %-5s | %8s %8s %8s %8s %8s %8s | %6s' % ('noise', 'proto', 'n', 'fx', 'fy', 'cx', 'cy', 'k1', 'k2', 'RMS'))
for noise in (0.0, 0.3, 1.0):
    for proto, n in (('C', 15), ('C', 8), ('B', 15)):
        dsyn = S.make_synthetic_set(proto, n, noise_px=noise, seed=2)
        o = my_zhang(dsyn['objpoints'], dsyn['imgpoints'], dsyn['image_size'])
        print('%-8.1f %-6s %-5d | %8.2f %8.2f %8.2f %8.2f %8.4f %8.4f | %6.3f   σ(f)=%.2f' % (noise, proto, n, o['K'][0, 0], o['K'][1, 1], o['K'][0, 2], o['K'][1, 2], o['dist'][0], o['dist'][1], o['rms'], o['std'][0]))
CK.check_full_pipeline(my_zhang); CK.check_full_pipeline(my_zhang, 'B');

## 2.7 OpenCV 와 비교 — 세트 C (2점)
`cv2.calibrateCameraExtended` (검증용 허용). 같은 모델(k₁ k₂, 접선 0, k₃ 0)로 충분히 수렴하면 비슷한 결과가 나와야 합니다. 입력 정밀도와 최적화 종료 조건이 다르므로 소수점 자릿수의 일치를 강제하지 말고, RMS·파라미터 차이·σ를 함께 확인하세요.
이 셀은 Part 0의 코너 파일만 있으면 scratch 구현과 독립적으로 실행할 수 있습니다.
기본 모델(5계수)과도 비교해 k₃ · p₁ p₂ 를 넣으면 f · σ 가 어떻게 되는지 적으세요 (모델 선택 — 과적합 주의).

In [ ]:
import cv2
# scratch가 실패해도 공통 설정과 Part 0 코너 파일만으로 실행 가능.
detC = load_corners_checked('corners_setC.npz'); W, H = detC['image_size']
objs = [o.astype(np.float32) for o in detC['objpoints']]; imgs = [m.astype(np.float32).reshape(-1, 1, 2) for m in detC['imgpoints']]
rows = []
for name, flags in [('k1k2 (same model)', cv2.CALIB_FIX_K3 | cv2.CALIB_ZERO_TANGENT_DIST), ('default 5-coef', 0)]:
    rms_cv, Kc, dc, rv, tv, sdi, sde, pve = cv2.calibrateCameraExtended(objs, imgs, (W, H), None, None, flags=flags)
    rows.append((name, Kc, dc.ravel(), rms_cv, sdi.ravel()))
print('%-20s %8s %8s %8s %8s %8s %8s %8s %6s' % ('', 'fx', 'fy', 'cx', 'cy', 'k1', 'k2', 'k3', 'RMS'))
for name, Kc, dc, r, sd in rows:
    print('%-20s %8.2f %8.2f %8.2f %8.2f %8.4f %8.4f %8.4f %6.3f  σ(f)=%.2f' % (name, Kc[0,0], Kc[1,1], Kc[0,2], Kc[1,2], dc[0], dc[1], dc[4] if len(dc) > 4 else 0, r, sd[0]))
Kc, dc, rms_cv, sd = rows[0][1], rows[0][2], rows[0][3], rows[0][4]
IO.save_json(IO.rpath('opencv_setC.json'), dict(K=Kc, dist=dc, rms=rms_cv, std=dict(fx=sd[0], fy=sd[1], cx=sd[2], cy=sd[3]), image_size=[W, H], scale=detC['scale'], source='opencv_setC', flags='FIX_K3|ZERO_TANGENT', K_default=rows[1][1], dist_default=rows[1][2], rms_default=rows[1][3]))
if all(n in globals() for n in ('K_z', 'dist_z', 'rms', 'std')):
    print('%-20s %8.2f %8.2f %8.2f %8.2f %8.4f %8.4f %8s %6.3f  σ(f)=%.2f' % ('scratch (ours)', K_z[0,0], K_z[1,1], K_z[0,2], K_z[1,2], dist_z[0], dist_z[1], '-', rms, std[0]))
    print('scratch − OpenCV: Δf = %.4f, %.4f  Δc = %.4f, %.4f  Δk1 = %.6f' % (K_z[0,0]-Kc[0,0], K_z[1,1]-Kc[1,1], K_z[0,2]-Kc[0,2], K_z[1,2]-Kc[1,2], dist_z[0]-dc[0]))
else:
    print('scratch 결과 없음 — OpenCV 결과를 독립 저장했습니다. 필요하면 아래 fallback을 사용하세요.')


## 2.8 세트 A · B · C 비교 (4점)
세 세트를 같은 파이프라인으로. 세트 A 는 **닫힌 해가 실패**할 수 있습니다(B 가 양정치가 아님 — Zhang §4 퇴화) — 그때는 세트 C 의 K 로 LM 을 시작하고(그 사실을 기록), σ(f) 를 보세요.

**보고서 질문 2.8** — (1) RMS 가 가장 낮은 세트는? (2) f 를 믿을 수 있는 세트는? (3) 그 근거 셋 (σ(f) · 커버리지 · 잔차의 공간 분포 · k 의 안정성 · 자세 다양성 중에서 — 품질 진단 체크리스트)

In [ ]:
summary = {}; o_ABC = {}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, s in zip(axes, ['setA', 'setB', 'setC']):
    d = dets[s]
    try:
        o = my_zhang(d['objpoints'], d['imgpoints'], d['image_size']); init = 'closed-form'
    except (ValueError, np.linalg.LinAlgError) as e:
        print(f'{s}: 닫힌 해 실패 ({e}) → 세트 C 의 K 로 LM 시작');
        o = my_zhang(d['objpoints'], d['imgpoints'], d['image_size'], K_init=K_z); init = 'K_init=setC'
    cov = Bd.coverage_map(d['imgpoints'], d['image_size'], ax=ax, title=f'{s}')
    o_ABC[s] = o
    summary[s] = dict(f=o['K'][0, 0], sigma_f=o['std'][0], rms=o['rms'], k1=o['dist'][0], k2=o['dist'][1], sigma_k1=o['std'][4], init=init,
                      empty_cells=int((cov == 0).sum()), max_per_view=max(o['per_view_rms']), n=len(d['names']))
    save_zhang(f'zhang_{s}.json', o['K'], o['dist'], o['std'], o['rms'], o['per_view_rms'], o['K0'], d, extra=dict(init=init, converged=o['converged'], termination=o['termination']))
fig.tight_layout()
print('%-6s %4s %9s %8s %7s %9s %9s %9s %6s %s' % ('set', 'n', 'f', 'σ(f)', 'RMS', 'k1', 'σ(k1)', 'k2', 'empty', 'init'))
for s, r in summary.items():
    print('%-6s %4d %9.2f %8.2f %7.3f %9.4f %9.4f %9.4f %6d %s' % (s, r['n'], r['f'], r['sigma_f'], r['rms'], r['k1'], r['sigma_k1'], r['k2'], r['empty_cells'], r['init']))
IO.save_json(IO.rpath('part2_ABC_summary.json'), summary)

In [ ]:
# 잔차의 공간 분포 — 세트별: 가장자리에서 체계적으로 커지는가? (재투영 오차의 함정)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, s in zip(axes, ['setA', 'setB', 'setC']):
    o, d = o_ABC[s], dets[s]
    resid = [m - project_points(o['K'], o['dist'], R, t, X) for R, t, X, m in zip(o['Rs'], o['ts'], d['objpoints'], d['imgpoints'])]
    Bd.residual_map(d['imgpoints'], resid, d['image_size'], ax=ax, title=f'{s} 관측 - 재투영 (RMS {o["rms"]:.3f})')
fig.tight_layout(); fig.savefig(IO.rpath('part2_residual_maps_ABC.png'), dpi=110)

## Fallback (구현에 실패한 경우에만)
Part 3–6 를 진행할 수 있도록 OpenCV 결과로 `K.json` · `dist.json` 을 쓸 수 있습니다. **이 셀을 실행하면 2.5 항목(9점) 은 0점, 2.2–2.4 는 셀프체크 통과 여부로 부분 점수**. 보고서에 명시해야 합니다.
실행 순서: **공통 설정 → 2.7 OpenCV 비교 셀 → 아래 셀**. 2.7은 scratch 변수나 TODO 구현을 요구하지 않습니다.

In [ ]:
USE_OPENCV_FALLBACK = False
if USE_OPENCV_FALLBACK:
    if not os.path.exists(IO.rpath('opencv_setC.json')):
        raise FileNotFoundError('먼저 2.7 OpenCV 비교 셀을 실행하세요. scratch 구현 없이 실행할 수 있습니다.')
    z = IO.load_json(IO.rpath('opencv_setC.json'))
    IO.save_K('K.json', np.array(z['K']), z['image_size'], 'opencv_fallback', std=z['std'], rms=z['rms'])
    IO.save_json(IO.rpath('dist.json'), dict(dist=z['dist'], model='opencv_k1k2', source='opencv_fallback'))
    print('OpenCV 결과를 K.json · dist.json 으로 저장했습니다 (감점 항목).')

## 보고서에 넣을 것 (Part 2, 슬라이드 5장)
1. 파이프라인 그림(닫힌 해 → k 초기화 → LM) + 합성 검증표(2.6)
2. 세트 C 결과: K · k · RMS · σ 표, scratch vs OpenCV 비교표(2.7)
3. 잔차 벡터장 + 장별 RMS (세트 C)
4. 세트 A · B · C 비교표 + 커버리지 맵 + 잔차 맵
5. 질문 2.8 의 답 (RMS 최저 세트 / f 를 믿을 세트 / 근거 셋)

In [ ]:
CK.run_all(globals());